# EXP H Threshold Exp

Threshold sweep notebook for: 0.50, 0.45, 0.40, 0.35, 0.30, 0.25, 0.20.

Uses EXP3 bundle artifacts where available and reports threshold-dependent tradeoffs.

In [ ]:
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

SEED = 42
THRESHOLDS = [0.50, 0.45, 0.40, 0.35, 0.30, 0.25, 0.20]

In [ ]:
ROOT = Path.cwd()
bundle_candidates = [
    ROOT / 'FINAL' / 'exp3_logreg_cat' / 'models' / 'best_model_bundle.joblib',
    ROOT / 'exp3_logreg_cat' / 'models' / 'best_model_bundle.joblib',
]
bundle_path = next((p for p in bundle_candidates if p.exists()), None)
if bundle_path is None:
    raise FileNotFoundError('best_model_bundle.joblib not found for exp3_logreg_cat')

bundle = joblib.load(bundle_path)
print('Loaded bundle:', bundle_path)
print('Bundle keys:', list(bundle.keys()) if isinstance(bundle, dict) else type(bundle))

In [ ]:
data_candidates = [
    ROOT / '0_MISC' / 'merged_clinical_leftjoin.csv',
    ROOT / 'merged_clinical_leftjoin.csv',
]
data_path = next((p for p in data_candidates if p.exists()), None)
if data_path is None:
    raise FileNotFoundError('merged_clinical_leftjoin.csv not found in expected locations')

df = pd.read_csv(data_path)

def infer_target_column(frame: pd.DataFrame):
    candidates = ['hypertension', 'htn', 'target', 'label', 'outcome']
    cols_lc = {c.lower(): c for c in frame.columns}
    for c in candidates:
        if c in cols_lc:
            return cols_lc[c]
    sbp = next((c for c in frame.columns if c.lower() in {'ave_sbp', 'sbp'}), None)
    dbp = next((c for c in frame.columns if c.lower() in {'ave_dbp', 'dbp'}), None)
    if sbp is not None and dbp is not None:
        frame = frame.copy()
        frame['Hypertension'] = (((pd.to_numeric(frame[sbp], errors='coerce') >= 140) | (pd.to_numeric(frame[dbp], errors='coerce') >= 90)).fillna(False)).astype(int)
        return 'Hypertension', frame
    return None, frame

target_col, df2 = infer_target_column(df)
if target_col is None:
    raise ValueError('Could not infer hypertension target column')

df2 = df2.dropna(subset=[target_col]).copy()
y_raw = df2[target_col]
y = pd.Series(LabelEncoder().fit_transform(y_raw.astype(str)), index=y_raw.index) if y_raw.nunique() != 2 else y_raw.astype(int)
X = df2.drop(columns=[target_col]).copy()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=SEED)
print('Test size:', len(X_test), '| positive rate:', float(np.mean(y_test)))

In [ ]:
if isinstance(bundle, dict):
    model = bundle.get('model', bundle.get('best_model', bundle.get('estimator')))
    preprocessor = bundle.get('preprocessor', bundle.get('transformer'))
else:
    model = bundle
    preprocessor = None

if model is None:
    raise ValueError('Model object not found in bundle')

if preprocessor is not None:
    X_test_model = preprocessor.transform(X_test)
else:
    X_test_model = X_test

if hasattr(model, 'predict_proba'):
    prob = np.asarray(model.predict_proba(X_test_model))[:, 1]
else:
    raw = np.asarray(model.predict(X_test_model)).astype(float)
    prob = np.clip(raw, 0.0, 1.0)

print('AUC:', roc_auc_score(y_test, prob))

In [ ]:
rows = []
for th in THRESHOLDS:
    y_hat = (prob >= th).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, y_hat, labels=[0, 1]).ravel()

    specificity = tn / (tn + fp) if (tn + fp) else np.nan
    npv = tn / (tn + fn) if (tn + fn) else np.nan

    rows.append({
        'threshold': th,
        'accuracy': accuracy_score(y_test, y_hat),
        'precision': precision_score(y_test, y_hat, zero_division=0),
        'recall': recall_score(y_test, y_hat, zero_division=0),
        'specificity': specificity,
        'f1': f1_score(y_test, y_hat, zero_division=0),
        'npv': npv,
        'tp': int(tp),
        'fp': int(fp),
        'tn': int(tn),
        'fn': int(fn),
    })

threshold_results = pd.DataFrame(rows).sort_values('threshold', ascending=False).reset_index(drop=True)
threshold_results

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(8, 5))
ax.plot(threshold_results['threshold'], threshold_results['recall'], marker='o', label='Recall')
ax.plot(threshold_results['threshold'], threshold_results['precision'], marker='o', label='Precision')
ax.plot(threshold_results['threshold'], threshold_results['specificity'], marker='o', label='Specificity')
ax.invert_xaxis()
ax.set_title('Threshold Trade-offs (EXP H)')
ax.set_xlabel('Threshold')
ax.set_ylabel('Metric Value')
ax.grid(alpha=0.3)
ax.legend()
plt.show()